## Theta Band Epoch-Level Variance Investigation

Load the baseline-corrected theta power for every participant and block.
Goal: find which epochs / ROIs / participants drive the inflated variance.

In [1]:
import polars as pl
import numpy as np
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go

BASE_ALL = Path("../EV_results/EV_l1")
ALL_PIDS = sorted([p.name for p in BASE_ALL.iterdir() if p.is_dir()])
print("Participants:", ALL_PIDS)


Participants: ['EV_002', 'EV_003', 'EV_004', 'EV_007', 'EV_008']


In [2]:
# --- Load and unpack baseline-corrected theta epoch files ---
# Each file (EV_XXX_eeg_psd_epochs_bsN_eeg_theta_bs.parquet) is a plot-spec parquet
# with fields: condition, x_data (ROI names), y_data (list of theta per ROI), y_var, ci_lower, ci_upper

flat_rows = []

for pid in ALL_PIDS:
    plots_dir = BASE_ALL / pid / "plots"
    if not plots_dir.exists():
        continue
    for bs in ["bs1", "bs2", "bs3"]:
        f = plots_dir / f"{pid}_eeg_psd_epochs_{bs}_eeg_theta_bs.parquet"
        if not f.exists():
            # Fallback: bare epoch file (may contain all bands)
            f = plots_dir / f"{pid}_eeg_psd_epochs_{bs}.parquet"
        if not f.exists():
            continue
        df = pl.read_parquet(f)
        for row in df.iter_rows(named=True):
            cond    = row.get("condition") or "?"
            rois    = row.get("x_data") or []
            vals    = row.get("y_data") or []
            vars_   = row.get("y_var") or [None] * len(vals)
            # vals may be [[band0_roi0, ...], [band1_roi0, ...]] or flat [roi0, roi1, ...]
            if isinstance(vals, list) and vals and isinstance(vals[0], list):
                # Multiple bands â€” take theta (assume last band or find by labels)
                labels = row.get("labels") or []
                theta_idx = next((i for i, l in enumerate(labels) if "theta" in str(l).lower()), -1)
                vals  = vals[theta_idx]  if theta_idx >= 0 else vals[0]
                vars_ = vars_[theta_idx] if (isinstance(vars_, list) and vars_ and isinstance(vars_[0], list)) else vars_
            if not isinstance(vars_, list):
                vars_ = [None] * len(vals)
            for roi, val, var in zip(rois, vals, vars_):
                flat_rows.append({
                    "participant": pid, "block": bs, "condition": str(cond),
                    "roi": str(roi), "theta": float(val) if val is not None else None,
                    "theta_var": float(var) if var is not None else None,
                })

theta_flat = pl.DataFrame(flat_rows).drop_nulls("theta")
print(f"Flat theta table: {theta_flat.shape}")
print(theta_flat.head(10))

Flat theta table: (15, 6)
shape: (10, 6)
┌─────────────┬───────┬───────────┬─────┬────────────┬────────────┐
│ participant ┆ block ┆ condition ┆ roi ┆ theta      ┆ theta_var  │
│ ---         ┆ ---   ┆ ---       ┆ --- ┆ ---        ┆ ---        │
│ str         ┆ str   ┆ str       ┆ str ┆ f64        ┆ f64        │
╞═════════════╪═══════╪═══════════╪═════╪════════════╪════════════╡
│ EV_002      ┆ bs1   ┆ NEG       ┆ NEG ┆ 153.041366 ┆ 36.823028  │
│ EV_002      ┆ bs2   ┆ NEU       ┆ NEU ┆ 105.199528 ┆ 0.666682   │
│ EV_002      ┆ bs3   ┆ POS       ┆ POS ┆ 241.531232 ┆ 295.384685 │
│ EV_003      ┆ bs1   ┆ NEG       ┆ NEG ┆ 285.9607   ┆ 23.615583  │
│ EV_003      ┆ bs2   ┆ NEU       ┆ NEU ┆ 549.442227 ┆ 592.389412 │
│ EV_003      ┆ bs3   ┆ POS       ┆ POS ┆ 244.656702 ┆ 65.570264  │
│ EV_004      ┆ bs1   ┆ NEG       ┆ NEG ┆ 504.170694 ┆ 45.676458  │
│ EV_004      ┆ bs2   ┆ NEU       ┆ NEU ┆ 548.062248 ┆ 46.000608  │
│ EV_004      ┆ bs3   ┆ POS       ┆ POS ┆ 534.246831 ┆ 92.865491  │
│ EV_00

In [3]:
# --- Boxplot: distribution per participant Ã— condition Ã— ROI ---
fig = px.box(
    theta_flat.to_pandas(),
    x="participant", y="theta", color="condition",
    facet_row="roi", facet_row_spacing=0.05,
    points="all", hover_data=["block"],
    title="Theta Power per Participant Ã— Condition Ã— ROI",
    labels={"theta": "Theta Power (ÂµVÂ²/Hz)"},
)
fig.update_layout(height=220 * theta_flat["roi"].n_unique() + 80)
fig.show()

In [4]:
# --- Outlier flagging: IQR + z-score per participant × condition × ROI ---
GROUP = ["participant", "condition", "roi"]

theta_scored = (
    theta_flat
    .with_columns([
        pl.col("theta").mean().over(GROUP).alias("_mu"),
        pl.col("theta").std().over(GROUP).alias("_sd"),
        pl.col("theta").quantile(0.25).over(GROUP).alias("_q1"),
        pl.col("theta").quantile(0.75).over(GROUP).alias("_q3"),
    ])
    .with_columns([
        pl.when(pl.col("_sd") > 0)
          .then((pl.col("theta") - pl.col("_mu")) / pl.col("_sd"))
          .otherwise(0.0).alias("z_score"),
        (pl.col("_q3") - pl.col("_q1")).alias("_iqr"),
    ])
    .with_columns([
        (
            (pl.col("z_score").abs() > 3.0) |
            (pl.col("theta") < pl.col("_q1") - 1.5 * pl.col("_iqr")) |
            (pl.col("theta") > pl.col("_q3") + 1.5 * pl.col("_iqr"))
        ).alias("outlier")
    ])
    .drop(["_mu", "_sd", "_q1", "_q3", "_iqr"])
)

# Summary table
outlier_summary = (
    theta_scored.group_by(GROUP)
    .agg([
        pl.len().alias("n"),
        pl.col("outlier").sum().alias("n_outliers"),
        pl.col("theta").mean().round(3).alias("mean_theta"),
        pl.col("theta").std().round(3).alias("std_theta"),
        pl.col("theta_var").mean().round(3).alias("mean_reported_var"),
    ])
    .with_columns((pl.col("n_outliers") / pl.col("n") * 100).round(1).alias("outlier_pct"))
    .sort(["participant", "roi", "condition"])
)
display(outlier_summary)


participant,condition,roi,n,n_outliers,mean_theta,std_theta,mean_reported_var,outlier_pct
str,str,str,u32,u32,f64,f64,f64,f64
"""EV_002""","""NEG""","""NEG""",1,0,153.041,null,36.823,0.0
"""EV_002""","""NEU""","""NEU""",1,0,105.2,null,0.667,0.0
"""EV_002""","""POS""","""POS""",1,0,241.531,null,295.385,0.0
"""EV_003""","""NEG""","""NEG""",1,0,285.961,null,23.616,0.0
"""EV_003""","""NEU""","""NEU""",1,0,549.442,null,592.389,0.0
…,…,…,…,…,…,…,…,…
"""EV_007""","""NEU""","""NEU""",1,0,157.781,null,11.451,0.0
"""EV_007""","""POS""","""POS""",1,0,155.299,null,17.092,0.0
"""EV_008""","""NEG""","""NEG""",1,0,698.404,null,19.009,0.0


In [5]:
# --- Scatter: all theta values, outliers in red ---
fig = px.strip(
    theta_scored.to_pandas(),
    x="participant", y="theta",
    color="outlier",
    facet_col="condition",
    facet_row="roi",
    color_discrete_map={True: "red", False: "steelblue"},
    hover_data=["block", "z_score"],
    title="Theta Epoch Values â€” Outliers in Red (|z|>3 or IQRÃ—1.5)",
    category_orders={"outlier": [False, True]},
)
fig.update_layout(height=230 * theta_scored["roi"].n_unique() + 60)
fig.show()

In [6]:
# --- Reported variance vs actual variance comparison ---
# Checks if the pipeline's reported y_var matches what we compute from y_data
variance_check = (
    theta_scored.group_by(GROUP)
    .agg([
        (pl.col("theta").var()).alias("computed_var"),
        pl.col("theta_var").first().alias("reported_var"),
    ])
    .with_columns(
        (pl.col("computed_var") - pl.col("reported_var")).abs().alias("var_discrepancy")
    )
    .sort("var_discrepancy", descending=True)
)
display(variance_check.head(15))


participant,condition,roi,computed_var,reported_var,var_discrepancy
str,str,str,f64,f64,f64
"""EV_008""","""NEG""","""NEG""",null,19.009106,null
"""EV_007""","""NEU""","""NEU""",null,11.451418,null
"""EV_003""","""NEG""","""NEG""",null,23.615583,null
"""EV_003""","""POS""","""POS""",null,65.570264,null
"""EV_004""","""NEU""","""NEU""",null,46.000608,null
…,…,…,…,…,…
"""EV_002""","""POS""","""POS""",null,295.384685,null
"""EV_007""","""POS""","""POS""",null,17.092095,null
"""EV_002""","""NEG""","""NEG""",null,36.823028,null


In [ ]:
# --- Worst outlier epochs: show block and which participant ---
worst = theta_scored.filter(pl.col("outlier")).sort("z_score", descending=True)
print(f"Total outlier data points: {worst.height}")
display(worst.select(["participant", "block", "condition", "roi", "theta", "z_score"]))


## Theta Band — Individual 30 s Epoch Outliers

The `_eeg_theta_bs.parquet` files hold **block-level means** (one value per block × ROI × condition).  
The actual individual epoch rows may live in the `eeg_cleaned_epochs_psdN_plot.parquet` files  
(one row per epoch, x_data = frequency bins, y_data = PSD values).  
This section probes those files, extracts the theta-band mean per epoch, and builds  
a scatter plot where every dot is one 30 s window — making it easy to spot noisy epochs.


In [7]:
# --- Probe: what do the eeg_cleaned_epochs_psdN_plot.parquet files contain? ---
import polars as pl
from pathlib import Path

BASE_ALL = Path("../EV_results/EV_l1")
sample_pid = next(p.name for p in BASE_ALL.iterdir() if p.is_dir())
plots_dir = BASE_ALL / sample_pid / "plots"

for psd_n in ["psd1", "psd2", "psd3"]:
    candidates = list(plots_dir.glob(f"*epochs_{psd_n}_plot.parquet"))
    if not candidates:
        continue
    df = pl.read_parquet(candidates[0])
    row0 = df.row(0, named=True)
    x = row0.get("x_data") or []
    y = row0.get("y_data") or []
    print(f"\n{candidates[0].name}")
    print(f"  rows={df.height}, cols={df.columns}")
    print(f"  x_data[0:5]={x[:5]}  (len={len(x)})")
    print(f"  y_data type={type(y[0]) if y else 'empty'}, len={len(y)}")
    if y and isinstance(y[0], list):
        print(f"  y_data[0][0:5]={y[0][:5]}")
    else:
        print(f"  y_data[0:5]={y[:5]}")
    if "condition" in df.columns:
        print(f"  conditions={df['condition'].to_list()}")
    if "labels" in df.columns:
        print(f"  labels={row0.get('labels')}")



EV_002_xdf4_aux_extr4_eeg_reref_filt_ica_eeg_cleaned_epochs_psd1_plot.parquet
  rows=1, cols=['condition', 'x_data', 'y_data', 'y_var', 'plot_type', 'x_label', 'y_label', 'y_ticks']
  x_data[0:5]=['theta', 'alpha', 'beta']  (len=3)
  y_data type=<class 'float'>, len=3
  y_data[0:5]=[115.95412508502466, 44.08308588184897, 25.411004459135103]
  conditions=['NEG']

EV_002_xdf4_aux_extr4_eeg_reref_filt_ica_eeg_cleaned_epochs_psd2_plot.parquet
  rows=1, cols=['condition', 'x_data', 'y_data', 'y_var', 'plot_type', 'x_label', 'y_label', 'y_ticks']
  x_data[0:5]=['theta', 'alpha', 'beta']  (len=3)
  y_data type=<class 'float'>, len=3
  y_data[0:5]=[94.7244525640151, 44.79537610255056, 24.765995967765235]
  conditions=['NEU']

EV_002_xdf4_aux_extr4_eeg_reref_filt_ica_eeg_cleaned_epochs_psd3_plot.parquet
  rows=1, cols=['condition', 'x_data', 'y_data', 'y_var', 'plot_type', 'x_label', 'y_label', 'y_ticks']
  x_data[0:5]=['alpha', 'beta', 'theta']  (len=3)
  y_data type=<class 'float'>, len=3
  

In [8]:
# --- Data-granularity check ---
# Confirm what theta_flat rows actually represent
print("theta_flat columns:", theta_flat.columns)
print("Unique blocks:", theta_flat["block"].unique().sort().to_list())
print("Unique ROIs:", theta_flat["roi"].unique().sort().to_list())
print(
    "\nRows per participant × condition × ROI:\n",
    theta_flat.group_by(["participant", "condition", "roi"])
    .agg(pl.len().alias("n"))
    .sort(["participant", "roi", "condition"])
    .head(12)
)
# Each row = one block (bs1/bs2/bs3) for a given participant × condition × ROI.
# bs1/bs2/bs3 are the three recording sessions; within each, multiple 30 s windows
# were averaged. theta_var = variance across those 30 s windows (within-block epoch var).


theta_flat columns: ['participant', 'block', 'condition', 'roi', 'theta', 'theta_var']
Unique blocks: ['bs1', 'bs2', 'bs3']
Unique ROIs: ['NEG', 'NEU', 'POS']

Rows per participant × condition × ROI:
 shape: (12, 4)
┌─────────────┬───────────┬─────┬─────┐
│ participant ┆ condition ┆ roi ┆ n   │
│ ---         ┆ ---       ┆ --- ┆ --- │
│ str         ┆ str       ┆ str ┆ u32 │
╞═════════════╪═══════════╪═════╪═════╡
│ EV_002      ┆ NEG       ┆ NEG ┆ 1   │
│ EV_002      ┆ NEU       ┆ NEU ┆ 1   │
│ EV_002      ┆ POS       ┆ POS ┆ 1   │
│ EV_003      ┆ NEG       ┆ NEG ┆ 1   │
│ EV_003      ┆ NEU       ┆ NEU ┆ 1   │
│ …           ┆ …         ┆ …   ┆ …   │
│ EV_004      ┆ NEU       ┆ NEU ┆ 1   │
│ EV_004      ┆ POS       ┆ POS ┆ 1   │
│ EV_007      ┆ NEG       ┆ NEG ┆ 1   │
│ EV_007      ┆ NEU       ┆ NEU ┆ 1   │
│ EV_007      ┆ POS       ┆ POS ┆ 1   │
└─────────────┴───────────┴─────┴─────┘


In [9]:
# --- Probe the broader EEG epoch files for channel/ROI-level theta ---
# Check: eeg_psd_epochs_bsN.parquet (no band suffix), eeg_cleaned_epochs_psdN_faipsd_fai.parquet

for sample_pid in ALL_PIDS[:2]:
    plots_dir = BASE_ALL / sample_pid / "plots"
    for fname in sorted(plots_dir.glob("*.parquet")):
        if "eeg" not in fname.name and "psd" not in fname.name:
            continue
        df = pl.read_parquet(fname)
        row0 = df.row(0, named=True)
        x = row0.get("x_data") or []
        y = row0.get("y_data") or []
        y0 = y[0] if y else None
        print(
            f"{fname.name[:70]:<70} | rows={df.height}"
            f" | x_len={len(x)} x[0]={x[0] if x else '?'!r}"
            f" | y_type={'list' if isinstance(y0, list) else 'scalar'}"
        )
    print()


EV_002_eeg_psd.parquet                                                 | rows=1 | x_len=3 x[0]='NEG' | y_type=list
EV_002_eeg_psd_epochs_bs1.parquet                                      | rows=1 | x_len=1 x[0]='NEG' | y_type=scalar
EV_002_eeg_psd_epochs_bs1_eeg_alpha_bs.parquet                         | rows=1 | x_len=1 x[0]='NEG' | y_type=scalar
EV_002_eeg_psd_epochs_bs1_eeg_beta_bs.parquet                          | rows=1 | x_len=1 x[0]='NEG' | y_type=scalar
EV_002_eeg_psd_epochs_bs1_eeg_theta_bs.parquet                         | rows=1 | x_len=1 x[0]='NEG' | y_type=scalar
EV_002_eeg_psd_epochs_bs2.parquet                                      | rows=1 | x_len=1 x[0]='NEU' | y_type=scalar
EV_002_eeg_psd_epochs_bs2_eeg_alpha_bs.parquet                         | rows=1 | x_len=1 x[0]='NEU' | y_type=scalar
EV_002_eeg_psd_epochs_bs2_eeg_beta_bs.parquet                          | rows=1 | x_len=1 x[0]='NEU' | y_type=scalar
EV_002_eeg_psd_epochs_bs2_eeg_theta_bs.parquet                    

In [10]:
# --- Focused probe: just eeg_psd_epochs files for one participant ---
sample_pid = ALL_PIDS[0]
plots_dir = BASE_ALL / sample_pid / "plots"
for fname in sorted(plots_dir.glob("*eeg_psd_epochs*.parquet")):
    df = pl.read_parquet(fname)
    row0 = df.row(0, named=True)
    x = row0.get("x_data") or []
    y = row0.get("y_data") or []
    y0 = y[0] if y else None
    conds = df["condition"].to_list() if "condition" in df.columns else "N/A"
    labels = row0.get("labels") or []
    print(f"\n{fname.name}")
    print(f"  rows={df.height}, x_len={len(x)}, x[0:4]={x[:4]}, labels={labels[:4]}")
    print(f"  y_type={'list' if isinstance(y0, list) else 'scalar'}, y_len={len(y)}")
    print(f"  conditions={conds}")
    if isinstance(y0, list):
        print(f"  y[0][0:4]={y0[:4]}")
    elif y:
        print(f"  y[0:4]={y[:4]}")



EV_002_eeg_psd_epochs_bs1.parquet
  rows=1, x_len=1, x[0:4]=['NEG'], labels=[]
  y_type=scalar, y_len=1
  conditions=['NEG']
  y[0:4]=[19.789505538875055]

EV_002_eeg_psd_epochs_bs1_eeg_alpha_bs.parquet
  rows=1, x_len=1, x[0:4]=['NEG'], labels=[]
  y_type=scalar, y_len=1
  conditions=['NEG']
  y[0:4]=[46.02561246526773]

EV_002_eeg_psd_epochs_bs1_eeg_beta_bs.parquet
  rows=1, x_len=1, x[0:4]=['NEG'], labels=[]
  y_type=scalar, y_len=1
  conditions=['NEG']
  y[0:4]=[19.789505538875055]

EV_002_eeg_psd_epochs_bs1_eeg_theta_bs.parquet
  rows=1, x_len=1, x[0:4]=['NEG'], labels=[]
  y_type=scalar, y_len=1
  conditions=['NEG']
  y[0:4]=[153.04136647183498]

EV_002_eeg_psd_epochs_bs2.parquet
  rows=1, x_len=1, x[0:4]=['NEU'], labels=[]
  y_type=scalar, y_len=1
  conditions=['NEU']
  y[0:4]=[105.19952834415132]

EV_002_eeg_psd_epochs_bs2_eeg_alpha_bs.parquet
  rows=1, x_len=1, x[0:4]=['NEU'], labels=[]
  y_type=scalar, y_len=1
  conditions=['NEU']
  y[0:4]=[40.83981107214035]

EV_002_eeg_psd

### Finding — `y_var` is a Bootstrap CI half-width, not variance

The pipeline chain is:
1. **`psd_analyzer`** computes per-epoch PSDs → stores `y_var = SEM` across 30 s epochs
2. **`bootstrap_analyzer`** resamples those epoch means → **overwrites** `y_var` with the bootstrap CI half-width (`max(|CI_upper − mean|, |mean − CI_lower|)`)
3. **`concatenating_processor`** merges conditions → passes `y_var` through unchanged

So `theta_var` in `theta_flat` is the **bootstrap CI half-width**, not a variance, not an epoch std.
The actual per-epoch values exist as bootstrap input but are **not saved to disk**.

The `variance_check` cell above was comparing two unrelated things:
- `computed_var` = variance of the 3 block means (across bs1/bs2/bs3)
- `reported_var` = bootstrap CI half-width *within* one block

**What the scatter below does:**
Plots each block's mean theta with the correct bootstrap CI as error bars —
wide CI = many noisy 30 s epochs or very few epochs → unreliable mean.


In [12]:
# --- Theta mean + bootstrap CI per participant × block (condition) ---
# theta_var = bootstrap CI half-width (NOT epoch variance or epoch std).
# ci_lower / ci_upper are the actual 95% CI bounds from the bootstrap step.
# Wide CI = many noisy 30 s epochs or very few epochs → extreme theta means unreliable.

ci_rows = []
for pid in ALL_PIDS:
    plots_dir = BASE_ALL / pid / "plots"
    if not plots_dir.exists():
        continue
    for bs in ["bs1", "bs2", "bs3"]:
        f = plots_dir / f"{pid}_eeg_psd_epochs_{bs}_eeg_theta_bs.parquet"
        if not f.exists():
            continue
        df = pl.read_parquet(f)
        for row in df.iter_rows(named=True):
            ci_rows.append({
                "participant": pid,
                "block": bs,
                "condition": str(row.get("condition") or "?"),
                "theta": float((row.get("y_data") or [None])[0] or 0),
                "ci_half": float((row.get("y_var") or [None])[0] or 0),
                "ci_lo": float((row.get("ci_lower") or [None])[0] or 0),
                "ci_hi": float((row.get("ci_upper") or [None])[0] or 0),
            })

theta_ci = pl.DataFrame(ci_rows).with_columns(
    (pl.col("ci_hi") - pl.col("ci_lo")).alias("ci_width")
)

# Flag blocks where CI width > 2× median (unstable bootstrap = noisy/few epochs)
med_ci = theta_ci["ci_width"].median()
wide_ci = theta_ci.filter(pl.col("ci_width") > 2 * med_ci)
if wide_ci.height > 0:
    print(f"Wide-CI blocks (ci_width > 2× median={med_ci:.1f} µV²/Hz):")
    display(wide_ci.select(["participant", "block", "condition", "theta", "ci_width"]).sort("ci_width", descending=True))

fig = px.scatter(
    theta_ci.to_pandas(),
    x="participant", y="theta",
    error_y="ci_half",
    color="block",
    facet_col="condition",
    facet_col_spacing=0.06,
    symbol="block",
    hover_data=["block", "ci_lo", "ci_hi", "ci_width"],
    title="Theta Power per Participant × Block<br>Error bars = bootstrap 95% CI (width ∝ epoch noise / n_epochs)",
    labels={"theta": "Theta Power (µV²/Hz)", "ci_half": "Bootstrap CI half-width"},
    color_discrete_sequence=px.colors.qualitative.Set2,
    category_orders={"block": ["bs1", "bs2", "bs3"]},
)
fig.update_traces(marker_size=9, error_y_thickness=2)
fig.update_layout(height=500)
fig.show()


Wide-CI blocks (ci_width > 2× median=47.5 µV²/Hz):


participant,block,condition,theta,ci_width
str,str,str,f64,f64
"""EV_003""","""bs2""","""NEU""",549.442227,901.662155
"""EV_002""","""bs3""","""POS""",241.531232,445.82948
"""EV_004""","""bs3""","""POS""",534.246831,140.1702
"""EV_003""","""bs3""","""POS""",244.656702,101.306127


In [15]:
# --- Which condition / block position drives wide theta CI? ---
# Hypothesis: the noisy condition is the SAME across participants
# (condition effect) vs. same BLOCK position (order/fatigue effect).

import plotly.graph_objects as go

df_pd = theta_ci.to_pandas()

# --- 1. CI width by condition (are NEG / NEU / POS inherently different?) ---
fig1 = px.strip(
    df_pd, x="condition", y="ci_width",
    color="participant",
    hover_data=["block", "theta", "ci_width"],
    title="Bootstrap CI Width by Condition<br>(wide = noisy epochs; if one condition is always wide → condition effect)",
    labels={"ci_width": "CI width (µV²/Hz)", "condition": "Condition"},
    category_orders={"condition": ["NEG", "NEU", "POS"]},
)
fig1.add_hline(y=float(med_ci), line_dash="dot", line_color="gray",
               annotation_text=f"median={med_ci:.0f}", annotation_position="right")
fig1.add_hline(y=float(2*med_ci), line_dash="dash", line_color="red",
               annotation_text="2× median", annotation_position="right")
fig1.update_layout(height=420)
fig1.show()

# --- 2. CI width by block position (is block 2 or 3 always worse?) ---
fig2 = px.strip(
    df_pd, x="block", y="ci_width",
    color="participant",
    hover_data=["condition", "theta", "ci_width"],
    title="Bootstrap CI Width by Block Position<br>(wide = noisy epochs; if one block is always wide → order/fatigue effect)",
    labels={"ci_width": "CI width (µV²/Hz)", "block": "Block"},
    category_orders={"block": ["bs1", "bs2", "bs3"]},
)
fig2.add_hline(y=float(med_ci), line_dash="dot", line_color="gray",
               annotation_text=f"median={med_ci:.0f}", annotation_position="right")
fig2.add_hline(y=float(2*med_ci), line_dash="dash", line_color="red",
               annotation_text="2× median", annotation_position="right")
fig2.update_layout(height=420)
fig2.show()

# --- 3. Pivot table: which participant × condition has high CI width ---
pivot = (
    theta_ci
    .select(["participant", "condition", "ci_width"])
    .pivot(on="condition", index="participant", values="ci_width", aggregate_function="mean")
)
print("Mean CI width per participant × condition (high = noisy):")
display(pivot)

# Summary: is the widest condition always the same one?
worst_cond_per_pid = (
    theta_ci
    .group_by(["participant", "condition"])
    .agg(pl.col("ci_width").mean().alias("mean_ci"))
    .sort(["participant", "mean_ci"], descending=[False, True])
    .group_by("participant", maintain_order=True)
    .first()
    .rename({"condition": "worst_condition", "mean_ci": "worst_mean_ci"})
)
print("\nCondition with widest CI per participant:")
display(worst_cond_per_pid.sort("participant"))


Mean CI width per participant × condition (high = noisy):


participant,NEG,NEU,POS
str,f64,f64,f64
"""EV_002""",56.850549,1.333364,445.82948
"""EV_003""",47.231166,901.662155,101.306127
"""EV_004""",84.124361,77.2382,140.1702
"""EV_007""",30.560435,19.212371,27.080989
"""EV_008""",37.388416,29.881881,47.476253



Condition with widest CI per participant:


participant,worst_condition,worst_mean_ci
str,str,f64
"""EV_002""","""POS""",445.82948
"""EV_003""","""NEU""",901.662155
"""EV_004""","""POS""",140.1702
"""EV_007""","""NEG""",30.560435
"""EV_008""","""POS""",47.476253


In [16]:
# --- Is the noisy condition always the same BLOCK POSITION? ---
# Conditions are counterbalanced across participants, so if the noise
# is order-driven (fatigue, drift, equipment warm-up), it will appear
# in the same bs1/bs2/bs3 slot regardless of which condition label is there.

print("Full theta CI per participant × block × condition:")
full_tbl = theta_ci.select(["participant","block","condition","theta","ci_width"]).sort(["participant","block"])
display(full_tbl)

# Which block position is the worst per participant?
worst_block_per_pid = (
    theta_ci
    .group_by(["participant", "block"])
    .agg(pl.col("ci_width").mean().alias("mean_ci"))
    .sort(["participant", "mean_ci"], descending=[False, True])
    .group_by("participant", maintain_order=True)
    .first()
    .rename({"block": "worst_block", "mean_ci": "worst_block_ci"})
)
print("\nBlock position with widest CI per participant:")
display(worst_block_per_pid.sort("participant"))

# Cross-tab: block position × condition label mapping
print("\nCondition order per participant (block→condition mapping):")
mapping = theta_ci.select(["participant","block","condition"]).sort(["participant","block"])
display(mapping.pivot(on="block", index="participant", values="condition"))


Full theta CI per participant × block × condition:


participant,block,condition,theta,ci_width
str,str,str,f64,f64
"""EV_002""","""bs1""","""NEG""",153.041366,56.850549
"""EV_002""","""bs2""","""NEU""",105.199528,1.333364
"""EV_002""","""bs3""","""POS""",241.531232,445.82948
"""EV_003""","""bs1""","""NEG""",285.9607,47.231166
"""EV_003""","""bs2""","""NEU""",549.442227,901.662155
…,…,…,…,…
"""EV_007""","""bs2""","""NEU""",157.78111,19.212371
"""EV_007""","""bs3""","""POS""",155.298533,27.080989
"""EV_008""","""bs1""","""NEG""",698.404045,37.388416



Block position with widest CI per participant:


participant,worst_block,worst_block_ci
str,str,f64
"""EV_002""","""bs3""",445.82948
"""EV_003""","""bs2""",901.662155
"""EV_004""","""bs3""",140.1702
"""EV_007""","""bs1""",30.560435
"""EV_008""","""bs3""",47.476253



Condition order per participant (block→condition mapping):


participant,bs1,bs2,bs3
str,str,str,str
"""EV_002""","""NEG""","""NEU""","""POS"""
"""EV_003""","""NEG""","""NEU""","""POS"""
"""EV_004""","""NEG""","""NEU""","""POS"""
"""EV_007""","""NEG""","""NEU""","""POS"""
"""EV_008""","""NEG""","""NEU""","""POS"""


## Theta Block Exclusion

**Finding**: Conditions are NOT counterbalanced — all participants ran NEG→NEU→POS in the same fixed order.  
The high-CI blocks are not consistent across block positions or conditions; they are **individual block failures** (likely EEG artifacts, electrode pop, or excessive movement during that specific 2-minute block).

**Exclusion criterion**: `ci_width > 2× median_across_all_blocks`  
This flags blocks where the bootstrap found that individual 30 s epoch means were so variable that a reliable estimate cannot be obtained.

| Participant | Block | Condition | θ mean | CI width | Reason |
|---|---|---|---|---|---|
| EV_002 | bs3 | POS | 242 | 446 | Extremely noisy epochs |
| EV_002 | bs2 | NEU | 105 | 1.3 | Suspiciously small CI → possibly single epoch |
| EV_003 | bs2 | NEU | 549 | 902 | Extremely noisy epochs |
| EV_004 | bs3 | POS | 534 | 140 | Elevated noise (>2× median) |

EV_007 and EV_008 are clean across all blocks.


In [18]:
# --- Theta exclusion: flag and remove unreliable blocks ---
# Criterion 1: ci_width > 2× median (very noisy epochs)
# Criterion 2: ci_width < 5 (suspiciously small → bootstrap collapsed, likely ≤1 epoch survived QC)

overall_median_ci = theta_ci["ci_width"].median()
threshold_high = 2.0 * overall_median_ci
threshold_low  = 5.0   # bootstrap CI < 5 on EEG PSD is physiologically implausible → degenerate block

theta_excluded = theta_ci.with_columns(
    pl.when(pl.col("ci_width") > threshold_high)
      .then(pl.lit("noisy (ci_width > 2×median)"))
      .when(pl.col("ci_width") < threshold_low)
      .then(pl.lit("degenerate (ci_width < 5)"))
      .otherwise(pl.lit("keep"))
      .alias("exclusion_reason")
)

excluded = theta_excluded.filter(pl.col("exclusion_reason") != "keep")
theta_clean = theta_excluded.filter(pl.col("exclusion_reason") == "keep")

print(f"Threshold noisy      : ci_width > {threshold_high:.1f}  (2× median={overall_median_ci:.1f})")
print(f"Threshold degenerate : ci_width < {threshold_low:.1f}")
print(f"\nExcluded blocks ({excluded.height}):")
display(excluded.select(["participant","block","condition","theta","ci_width","exclusion_reason"]).sort("ci_width", descending=True))
print(f"\nClean blocks remaining: {theta_clean.height} / {theta_ci.height}")

# --- Per-condition coverage after exclusion ---
print("\nCoverage per condition after exclusion:")
display(
    theta_excluded
    .group_by("condition")
    .agg(
        (pl.col("exclusion_reason") == "keep").sum().alias("n_clean"),
        pl.len().alias("n_total"),
    )
    .with_columns(((pl.col("n_clean") / pl.col("n_total")) * 100).round(0).alias("pct_clean"))
    .sort("condition")
)

# --- Plot clean data ---
import plotly.express as px

fig = px.scatter(
    theta_clean.to_pandas(),
    x="condition", y="theta",
    error_y="ci_half",
    color="participant",
    symbol="block",
    hover_data=["block","participant","ci_width"],
    title=f"Theta Power — Clean Blocks Only (excluded {excluded.height} noisy/degenerate blocks)<br>Error bars = bootstrap 95% CI",
    labels={"theta": "Theta Power (µV²/Hz)", "condition": "Condition"},
    category_orders={"condition": ["NEG","NEU","POS"]},
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_traces(marker_size=10, error_y_thickness=2)
fig.update_layout(height=480)
fig.show()

# Summary statistics on clean data
print("\nMean theta per condition (clean blocks only):")
display(
    theta_clean
    .group_by("condition")
    .agg(
        pl.col("theta").mean().round(1).alias("mean_theta"),
        pl.col("theta").std().round(1).alias("std_theta"),
        pl.col("ci_width").mean().round(1).alias("mean_ci_width"),
        pl.len().alias("n_blocks"),
    )
    .sort("condition")
)


Threshold noisy      : ci_width > 95.0  (2× median=47.5)
Threshold degenerate : ci_width < 5.0

Excluded blocks (5):


participant,block,condition,theta,ci_width,exclusion_reason
str,str,str,f64,f64,str
"""EV_003""","""bs2""","""NEU""",549.442227,901.662155,"""noisy (ci_width > 2×median)"""
"""EV_002""","""bs3""","""POS""",241.531232,445.82948,"""noisy (ci_width > 2×median)"""
"""EV_004""","""bs3""","""POS""",534.246831,140.1702,"""noisy (ci_width > 2×median)"""
"""EV_003""","""bs3""","""POS""",244.656702,101.306127,"""noisy (ci_width > 2×median)"""
"""EV_002""","""bs2""","""NEU""",105.199528,1.333364,"""degenerate (ci_width < 5)"""



Clean blocks remaining: 10 / 15

Coverage per condition after exclusion:


condition,n_clean,n_total,pct_clean
str,u32,u32,f64
"""NEG""",5,5,100.0
"""NEU""",3,5,60.0
"""POS""",2,5,40.0



Mean theta per condition (clean blocks only):


condition,mean_theta,std_theta,mean_ci_width,n_blocks
str,f64,f64,f64,u32
"""NEG""",357.9,239.0,51.2,5
"""NEU""",477.1,290.5,42.1,3
"""POS""",458.9,429.3,37.3,2


---
## Pipeline-wide `y_var` Audit

### What `y_var` actually means per module (from source code)

| Module | `y_var` written as | Correct? |
|---|---|---|
| `psd_analyzer` | **SEM** (`std / sqrt(n_epochs)`) across 30 s epochs | ✅ Consistent |
| `group_analyzer` (`_plot.parquet`) | **SEM** (`std / sqrt(n_epochs)`) across epochs per ROI | ✅ Consistent |
| `amplitude_analyzer` (with `y_lim`) | **SEM** across epochs | ✅ Consistent |
| `interval_analyzer` (HRV) | **SEM** across epochs | ✅ Consistent |
| `plv_analyzer` | **SEM** of PLV across epoch pairs | ✅ Consistent |
| `waveform_analyzer` | **SEM** of waveform across epochs | ✅ Consistent |
| `bootstrap_analyzer` | **CI half-width** `max(|CI_hi−mean|, |mean−CI_lo|)` — overwrites SEM | ⚠️ Different meaning |
| `relative_analyzer` | Passes through `y_var` **unchanged** from input (CI half-width or SEM) | ⚠️ Propagates whatever was there |
| `asymmetry_analyzer` | **SEM** propagated via error propagation from input SEM | ✅ Consistent if input was SEM |
| `concatenating_processor` | Passes through `y_var` **unchanged** from each input file | ⚠️ Merges without harmonising |

### The actual problem

The issue is **not** that variance is re-propagated incorrectly — each module does its own thing correctly. The problem is that `bootstrap_analyzer` runs **after** `amplitude_analyzer` / `group_analyzer` / `psd_analyzer` in the pipeline, and replaces `y_var = SEM` with `y_var = bootstrap CI half-width`.

These are numerically very different:
- **SEM** ≈ `std / sqrt(n)` — shrinks with more epochs
- **Bootstrap CI half-width** ≈ `1.96 × boot_SE` — also shrinks with more epochs, but reflects resampling noise

The extreme values (EV_003/bs2 theta=549, CI_width=902) mean the bootstrap found the epoch means were so variable that even after 10 000 resamples the CI spanned nearly the full range — confirming that block's individual 30 s epochs had wildly different theta values.

**For fNIRS**, the same chain applies via `amplitude_analyzer` → `bootstrap_analyzer` → `concatenating_processor`. The `hbc_var` in `fnirs_flat` is also a bootstrap CI half-width, not a variance. The `CV%` computed as `sqrt(hbc_var) / mean × 100` in the CV table is therefore also wrong — it's `CI_half_width_sqrt / mean`, which is meaningless.

The cell below audits the actual `y_var` values in the output files across all modalities.


In [13]:
# --- y_var audit: scan all plot-ready parquets for one participant ---
# For each file: show what y_var holds, whether a bootstrap step ran on it,
# and flag cases where y_var > y_data (CI half-width exceeds the mean itself)

import polars as pl
from pathlib import Path

sample_pid = ALL_PIDS[0]
plots_dir = BASE_ALL / sample_pid / "plots"

audit_rows = []
for f in sorted(plots_dir.glob("*.parquet")):
    df = pl.read_parquet(f)
    if "y_data" not in df.columns or "y_var" not in df.columns:
        continue
    row0 = df.row(0, named=True)
    y_data = row0.get("y_data") or []
    y_var  = row0.get("y_var")  or []
    ci_lo  = row0.get("ci_lower") or []
    ci_hi  = row0.get("ci_upper") or []

    # Flatten nested lists to first scalar values
    def first_scalar(v):
        while isinstance(v, list):
            if not v: return None
            v = v[0]
        return v

    y0  = first_scalar(y_data)
    v0  = first_scalar(y_var)
    clo = first_scalar(ci_lo)
    chi = first_scalar(ci_hi)

    has_ci = clo is not None and chi is not None
    is_bootstrap = "_bs" in f.name or ("_bs1" in f.name)
    # CI half-width > |mean| => extreme relative uncertainty
    flag = (v0 is not None and y0 is not None and y0 != 0 and abs(v0) > abs(y0))

    audit_rows.append({
        "file": f.name[:65],
        "mean_y0": round(y0, 3) if y0 is not None else None,
        "y_var_0": round(v0, 3) if v0 is not None else None,
        "has_CI":  has_ci,
        "is_bs":   is_bootstrap,
        "flag_var_gt_mean": flag,
    })

audit_df = pl.DataFrame(audit_rows)
flagged = audit_df.filter(pl.col("flag_var_gt_mean"))
print(f"\n{sample_pid}: {audit_df.height} plot-ready files, {flagged.height} with |y_var| > |mean|:\n")
display(flagged)

print("\nFull audit (first 40 rows):")
display(audit_df.head(40))



EV_002: 74 plot-ready files, 13 with |y_var| > |mean|:



file,mean_y0,y_var_0,has_CI,is_bs,flag_var_gt_mean
str,f64,f64,bool,bool,bool
"""EV_002_eeg_psd_epochs_bs3.parq…",241.531,295.385,true,true,true
"""EV_002_eeg_psd_epochs_bs3_eeg_…",241.531,295.385,true,true,true
"""EV_002_fnirsagg.parquet""",3.92,7.48,true,false,true
"""EV_002_hbc_fai.parquet""",-2.953,25.202,false,false,true
"""EV_002_xdf3_fnirs_log_tddr_reg…",3.92,7.48,true,true,true
…,…,…,…,…,…
"""EV_002_xdf3_fnirs_log_tddr_reg…",9.508,20.966,false,true,true
"""EV_002_xdf3_fnirs_log_tddr_reg…",-5.132,15.017,true,true,true
"""EV_002_xdf3_fnirs_log_tddr_reg…",-5.132,15.017,true,true,true



Full audit (first 40 rows):


file,mean_y0,y_var_0,has_CI,is_bs,flag_var_gt_mean
str,f64,f64,bool,bool,bool
"""EV_002_alpha.parquet""",46.026,4.523,true,false,false
"""EV_002_be7.parquet""",1.0,0.0,false,false,false
"""EV_002_beta.parquet""",19.79,2.215,true,false,false
"""EV_002_ea11.parquet""",1.0,0.0,false,false,false
"""EV_002_eda.parquet""",109.527,7.623,true,false,false
…,…,…,…,…,…
"""EV_002_txt_tree_ea113.parquet""",1.0,0.0,false,false,false
"""EV_002_txt_tree_ea113_ea11.par…",1.0,0.0,false,false,false
"""EV_002_txt_tree_panas.parquet""",3.0,0.0,false,false,false


In [14]:
# --- Deep-dive: flagged non-bootstrap files (hbc_fai, fnirsagg) ---
# These have y_var > mean but no bootstrap suffix — what is y_var there?

for fname in ["EV_002_fnirsagg.parquet", "EV_002_hbc_fai.parquet"]:
    f = plots_dir / fname
    if not f.exists():
        continue
    df = pl.read_parquet(f)
    row0 = df.row(0, named=True)
    print(f"\n=== {fname} ===")
    print(f"  columns: {df.columns}")
    print(f"  x_data:  {row0.get('x_data')}")
    print(f"  y_data:  {row0.get('y_data')}")
    print(f"  y_var:   {row0.get('y_var')}")
    print(f"  ci_lower:{row0.get('ci_lower')}")
    print(f"  ci_upper:{row0.get('ci_upper')}")
    print(f"  labels:  {row0.get('labels')}")

# Check: fnirsagg is the concatenated bootstrap output across the 3 conditions
# y_var here = list of CI half-widths, one per condition
# It's correctly a bootstrap CI — just a large one because fNIRS HbC has high epoch noise
print("\n--- Conclusion ---")
print("All flagged files fall into two categories:")
print("  1. Bootstrap output (is_bs=True): y_var = CI half-width. EXPECTED to be large for noisy signals.")
print("  2. fnirsagg / hbc_fai (is_bs=False but downstream of bootstrap): y_var = CI half-width passed through.")
print("  There is NO module writing actual statistical variance (std²) into y_var.")
print("  The naming 'y_var' is misleading — it is always SEM or CI half-width, never variance.")



=== EV_002_fnirsagg.parquet ===
  columns: ['x_data', 'plot_type', 'x_label', 'y_label', 'y_ticks', 'y_data', 'y_var', 'ci_lower', 'ci_upper', 'labels']
  x_data:  ['Left PFC', 'Right PFC', 'Left Parietal', 'Right Parietal']
  y_data:  [[3.9197190276635623, 0.96661083470323, -2.2814173472651196, 4.472131609135057], [-9.347991819856741, 0.16011078852166172, -4.845712422049099, 2.94011297855723], [-5.132223781152277, -3.5773534006244776, -7.346814777176108, -2.5103316766385073]]
  y_var:   [[7.479819686337198, 24.066563007926575, 9.944350638924071, 1.4653003856544875], [5.715610837698216, 20.171399096161508, 3.076573195720375, 1.6332827059422066], [15.017031764900796, 16.656994248676277, 15.690987832864373, 2.137895594961437]]
  ci_lower:[[-3.5151085806389766, -19.470381250479598, -12.22576798618919, 3.0068312234805696], [-15.063602657554958, -20.011288307639845, -7.922285617769473, 1.3068302726150234], [-19.704040110908025, -15.335395473379466, -16.45365108591235, -4.208228606533511]]


## fNIRS â€” Why is the Variance so Large?

The aggregated fNIRS results (`fnirsagg`) sometimes show CV% > 500%, meaning the standard
deviation exceeds the mean by a factor of 5. This section loads the per-block
HbC epoch files and traces where the variance is coming from:

1. Is it specific **participants** or all of them?
2. Is it specific **blocks** (one bad recording block)?
3. Is it specific **ROIs / channels**?
4. Does it relate to the **number of epochs** per condition (small n â†’ unstable bootstrap)?

In [ ]:
# --- Load per-block HbC epoch data across all participants ---
# Files: EV_XXX_xdf3_fnirs_log_tddr_regr_lin_filt_epochs_hbcN_fnirshbc_bsM.parquet
# These are plot-spec parquets: one row per condition with x_data=ROI names, y_data=HbC per ROI

fnirs_rows = []

for pid in ALL_PIDS:
    plots_dir = BASE_ALL / pid / "plots"
    if not plots_dir.exists():
        continue
    for hbc in ["hbc1", "hbc2", "hbc3"]:
        for bs in ["bs1", "bs2", "bs3"]:
            fname = f"{pid}_xdf3_fnirs_log_tddr_regr_lin_filt_epochs_{hbc}_fnirshbc_{bs}.parquet"
            f = plots_dir / fname
            if not f.exists():
                continue
            df = pl.read_parquet(f)
            for row in df.iter_rows(named=True):
                cond  = row.get("condition") or "?"
                rois  = row.get("x_data") or []
                vals  = row.get("y_data") or []
                vars_ = row.get("y_var") or [None]*len(vals)
                cis_lo = row.get("ci_lower") or [None]*len(vals)
                cis_hi = row.get("ci_upper") or [None]*len(vals)
                for roi, val, var, clo, chi in zip(rois, vals, vars_, cis_lo, cis_hi):
                    fnirs_rows.append({
                        "participant": pid, "block": bs, "hbc_set": hbc,
                        "condition": str(cond), "roi": str(roi),
                        "hbc": float(val)  if val  is not None else None,
                        "hbc_var":  float(var)  if var  is not None else None,
                        "ci_lo": float(clo) if clo is not None else None,
                        "ci_hi": float(chi) if chi is not None else None,
                    })

fnirs_flat = pl.DataFrame(fnirs_rows).drop_nulls("hbc")
print(f"fNIRS flat table: {fnirs_flat.shape}")
print(fnirs_flat.head(6))

In [ ]:
# --- CV% table: who has the worst relative variance? ---
fnirs_cv = (
    fnirs_flat
    .with_columns([
        pl.col("hbc_var").sqrt().alias("hbc_std"),
    ])
    .with_columns([
        pl.when(pl.col("hbc").abs() > 0.01)
          .then((pl.col("hbc_std") / pl.col("hbc").abs() * 100))
          .otherwise(None)
          .alias("cv_pct")
    ])
    .group_by(["participant", "roi", "condition"])
    .agg([
        pl.col("hbc").mean().round(3).alias("mean_hbc"),
        pl.col("hbc_var").mean().round(3).alias("mean_var"),
        pl.col("hbc_std").mean().round(3).alias("mean_std"),
        pl.col("cv_pct").mean().round(1).alias("mean_cv_pct"),
        pl.len().alias("n_blocks"),
    ])
    .sort("mean_cv_pct", descending=True, nulls_last=True)
)
display(fnirs_cv.head(30))


In [ ]:
# --- Block-level strip: HbC per ROI, all participants
# If one block drives the variance, it will appear as an isolated point
fig = px.strip(
    fnirs_flat.to_pandas(),
    x="roi", y="hbc",
    color="participant",
    facet_col="condition",
    facet_row="hbc_set",
    hover_data=["block", "hbc_var"],
    title="fNIRS HbC per Block â€” Each Dot is One Block's Mean",
    labels={"hbc": "HbC change (ÂµM)"},
)
fig.update_layout(height=280 * fnirs_flat["hbc_set"].n_unique() + 80)
fig.show()

In [ ]:
# --- CI width as a proxy for within-block epoch variance ---
# Wide CI = high epoch-to-epoch variability within that block
fnirs_flat2 = fnirs_flat.with_columns(
    (pl.col("ci_hi") - pl.col("ci_lo")).alias("ci_width")
).drop_nulls("ci_width")

fig = px.box(
    fnirs_flat2.to_pandas(),
    x="roi", y="ci_width",
    color="participant",
    facet_col="condition",
    points="all",
    hover_data=["block", "hbc_set"],
    title="fNIRS CI Width per Block (proxy for within-block epoch variance)",
    labels={"ci_width": "CI width (ÂµM)"},
)
fig.update_layout(height=400)
fig.show()

In [ ]:
# --- Cross-block variance: how much do blocks differ from each other?
# High cross-block variance means the three ~2-min segments of the rating images
# gave very different mean HbC — indicative of slow drift, not stimulus response.

block_var = (
    fnirs_flat
    .group_by(["participant", "hbc_set", "condition", "roi"])
    .agg([
        pl.col("hbc").mean().alias("mean_across_blocks"),
        pl.col("hbc").std().alias("across_block_std"),
        pl.col("hbc").min().alias("min_block"),
        pl.col("hbc").max().alias("max_block"),
        pl.len().alias("n_blocks"),
    ])
    .with_columns(
        (pl.col("max_block") - pl.col("min_block")).alias("block_range")
    )
    .sort("across_block_std", descending=True, nulls_last=True)
)
display(block_var.head(20))


In [ ]:
# --- Heatmap: across-block std per participant Ã— ROI (averaged across conditions)
import plotly.graph_objects as go

pivot = (
    block_var
    .group_by(["participant", "roi"])
    .agg(pl.col("across_block_std").mean().alias("mean_std"))
    .sort(["participant", "roi"])
)

participants_list = pivot["participant"].unique().sort().to_list()
rois_list         = pivot["roi"].unique().sort().to_list()

z_matrix = []
for pid in participants_list:
    row_vals = []
    for roi in rois_list:
        val = pivot.filter((pl.col("participant") == pid) & (pl.col("roi") == roi))["mean_std"]
        row_vals.append(float(val[0]) if val.len() > 0 else None)
    z_matrix.append(row_vals)

fig = go.Figure(go.Heatmap(
    z=z_matrix, x=rois_list, y=participants_list,
    colorscale="Reds",
    colorbar_title="Std (ÂµM)",
    text=[[f"{v:.2f}" if v is not None else "" for v in row] for row in z_matrix],
    texttemplate="%{text}",
))
fig.update_layout(
    title="fNIRS Cross-Block Std per Participant Ã— ROI<br>(high = drifts between blocks, likely systemic noise)",
    xaxis_title="ROI", yaxis_title="Participant",
    height=300,
)
fig.show()

## fNIRS — Within-Block Epoch Spread (30 s windows)

The pipeline stores **per-block means + variance** (`hbc_var`), not individual epoch rows.  
`hbc_var` is the variance *across* the 30 s sliding-window epochs within that block, so  
`sqrt(hbc_var)` = within-block epoch std — the closest proxy to the individual epoch scatter.

The scatter below plots each block as a point (mean ± 1 std), making it easy to see  
which participant/block combinations have wildly spread epochs.


In [ ]:
# --- Within-block epoch spread: mean ± sqrt(hbc_var) per block ---
# Each point = one block's mean HbC; error bars = within-block epoch std
# Large error bars = high epoch-to-epoch variability within that 2-min block

fnirs_spread = fnirs_flat.with_columns(
    pl.col("hbc_var").sqrt().alias("epoch_std")
).drop_nulls("epoch_std")

fig = px.scatter(
    fnirs_spread.to_pandas(),
    x="participant", y="hbc",
    error_y="epoch_std",
    color="hbc_set",
    symbol="block",
    facet_col="condition",
    facet_row="roi",
    facet_row_spacing=0.06,
    hover_data=["block", "hbc_set", "epoch_std"],
    title="fNIRS HbC — Block Mean ± Within-Block Epoch Std (30 s windows)",
    labels={"hbc": "HbC change (µM)", "epoch_std": "Epoch std (µM)"},
    category_orders={"block": ["bs1", "bs2", "bs3"]},
)
fig.update_traces(marker_size=7)
fig.update_layout(height=260 * fnirs_spread["roi"].n_unique() + 80)
fig.show()


---
## Pipeline Bug Summary & Fixes Applied

### Bug 1 — fNIRS: short-separation channels included as regular channels (CRITICAL)

**Two interacting bugs** caused completely invalid fNIRS data:

**1a. Wrong MBLL pairing (`fnirs_channels = '1:101:2'`)**  
The NIRx .fif has channels ordered as **all wl=0 first (pos 1-25), then all wl=1 (pos 26-50)**.  
`1:101:2` → selects positions 1,3,5,...,99 = all *odd* positions across all wavelength groups.  
The MBLL processor then applies the 2×2 matrix to *consecutive pairs* — but consecutive odd positions
are two **different SD pairs at the same wavelength**, not the same SD pair at two wavelengths.  
Result: MBLL output is meaningless (differences of two 760 nm signals, not HbO/HbR unmixing).

**1b. Short channels included by the odd-index selection**  
Positions 3, 9, 15, 21, 27, ... are short-detector channels (D8-D15) and fall on odd indices,
so they were included in the 50-channel MBLL input.

**Fix applied**: `fnirs_channels` changed to hardcoded interleaved list:  
`1,26, 2,27, 4,29, 5,30, ...` (17 long-detector SD pairs, each as (wl=0, wl=1) consecutive pair).  
→ 34 channels, MBLL applies correctly, no short channels, outputs valid HbO/HbR.

---

**1c. `regression_processor.py` regex mismatch**  
The short-channel detection regex `r'(^s\d+\b)|short|_sd|_short'` does not match NIRx channel names
like `'1-8:3-0'` (Source 1, Detector 8, short). The regression step was silently skipped for all
participants (`[regression] Warning: No short channels detected, skipping regression`).  
With the `fnirs_channels` fix, short channels no longer reach the regression step anyway.  
**Fix applied**: regex extended with `|-(?:8|9|1[0-5]):` to match NIRx short detector names.

---

### Bug 2 — EEG FAI: psd_fai y_var = 0.0 for all participants/conditions

The FAI asymmetry is computed from raw per-epoch PSD data (columns: `condition, epoch_id, channel, band, power`).  
`asymmetry_analyzer.py` falls back to `sem = 0.0` when no pre-computed `sem` column is present.  
Result: the frontal alpha asymmetry plot shows bars with **no visible error bars** (zero-height).

**Fix applied**: when per-epoch data is available (`epoch_id` column present), compute SEM as  
`std(power_epochs) / sqrt(n_epochs)` per channel before propagating through the log-asymmetry formula.

---

### Bug 3 — HRV/EDA variance: data is present, rendering is correct

After investigation, HRV and EDA **do** have bootstrap CI data:
- `hrv.parquet`: y_var = [CI_half_NEG, CI_half_NEU, CI_half_POS] (bootstrap 95% CI half-widths)
- `eda.parquet`: same structure with smaller relative uncertainty

The bar chart error bars ARE rendered by the interactive plotter. See the cell below to verify.

---

### What to do: pipeline re-run required for fNIRS and FAI fixes

All three code fixes are already applied to the source files. The user re-runs the pipeline.  
The fNIRS results (log → tddr → regression → MBLL → filter → epoch → bootstrap chain)  
and the EEG FAI results need to be regenerated from the corrected code.


---
## HRV, EDA, EEG-FAI — Variance Verification and Visualisation

Load the bootstrap-aggregated outputs for all three modalities to confirm CI data is present
and visualise condition-level means with 95% CI error bars.


In [19]:
# --- Load HRV, EDA, and EEG-FAI bootstrap results for all participants ---
# All three modalities use the same plot-spec structure:
#   y_data  = [mean_NEG, mean_NEU, mean_POS]
#   y_var   = [CI_half_NEG, CI_half_NEU, CI_half_POS]  (bootstrap 95% CI half-width)
#   ci_lower/ci_upper = full CI bounds
#   x_data  = ['NEG', 'NEU', 'POS']

modal_rows = []

for pid in ALL_PIDS:
    plots_dir = BASE_ALL / pid / "plots"
    if not plots_dir.exists():
        continue
    for modal, fname_tpl in [
        ("HRV",  f"{pid}_hrv.parquet"),
        ("EDA",  f"{pid}_eda.parquet"),
        ("FAI",  f"{pid}_psd_fai.parquet"),
    ]:
        f = plots_dir / fname_tpl
        if not f.exists():
            continue
        df = pl.read_parquet(f)
        row0 = df.row(0, named=True)
        conds  = row0.get("x_data") or []
        vals   = row0.get("y_data") or []
        vars_  = row0.get("y_var")  or [None] * len(vals)
        cis_lo = row0.get("ci_lower") or [None] * len(vals)
        cis_hi = row0.get("ci_upper") or [None] * len(vals)
        for cond, val, var, clo, chi in zip(conds, vals, vars_, cis_lo, cis_hi):
            modal_rows.append({
                "participant": pid,
                "modality": modal,
                "condition": str(cond),
                "mean": float(val)  if val  is not None else None,
                "ci_half": float(var)  if var  is not None else None,
                "ci_lo": float(clo) if clo is not None else None,
                "ci_hi": float(chi) if chi is not None else None,
            })

modal_flat = pl.DataFrame(modal_rows).drop_nulls("mean")
modal_flat = modal_flat.with_columns(
    (pl.col("ci_hi") - pl.col("ci_lo")).alias("ci_width")
)
print(f"Loaded {modal_flat.height} rows across HRV / EDA / EEG-FAI")
print(modal_flat.group_by("modality").agg(
    pl.len().alias("n_values"),
    (pl.col("ci_half") == 0).sum().alias("n_zero_ci"),
    pl.col("ci_half").mean().round(4).alias("mean_ci_half"),
))


Loaded 45 rows across HRV / EDA / EEG-FAI
shape: (3, 4)
┌──────────┬──────────┬───────────┬──────────────┐
│ modality ┆ n_values ┆ n_zero_ci ┆ mean_ci_half │
│ ---      ┆ ---      ┆ ---       ┆ ---          │
│ str      ┆ u32      ┆ u32       ┆ f64          │
╞══════════╪══════════╪═══════════╪══════════════╡
│ EDA      ┆ 15       ┆ 0         ┆ 7.0458       │
│ HRV      ┆ 15       ┆ 0         ┆ 51.9126      │
│ FAI      ┆ 15       ┆ 12        ┆ NaN          │
└──────────┴──────────┴───────────┴──────────────┘


In [20]:
# --- Plot: HRV, EDA, EEG-FAI bar with bootstrap CI per participant ---
# Note: EEG-FAI (psd_fai) has ci_half = 0 in current data (Bug 2 — fix applied, needs pipeline rerun).
# HRV and EDA have valid bootstrap CIs.

for modal in ["HRV", "EDA", "FAI"]:
    sub = modal_flat.filter(pl.col("modality") == modal).to_pandas()
    if sub.empty:
        print(f"No data for {modal}")
        continue

    fig = px.scatter(
        sub,
        x="condition", y="mean",
        error_y="ci_half",
        color="participant",
        symbol="participant",
        hover_data=["ci_lo", "ci_hi", "ci_width"],
        title=f"{modal} — Bootstrap 95% CI per Participant × Condition"
              + (" [NOTE: ci=0, Bug 2 still active — rerun pipeline]" if modal == "FAI" else ""),
        labels={"mean": sub["condition"].iloc[0] if False else
                ("RMSSD (ms)" if modal == "HRV" else
                 "EDA amplitude (µS)" if modal == "EDA" else "FAI (log ratio)")},
        category_orders={"condition": ["NEG", "NEU", "POS"]},
        color_discrete_sequence=px.colors.qualitative.Set2,
    )
    fig.update_traces(marker_size=10, error_y_thickness=2)
    fig.update_layout(height=450)
    fig.show()

# Summary: is ci_half = 0 for FAI?
print("\nFAI variance check (should be 0 in current run, non-zero after rerun):")
display(modal_flat.filter(pl.col("modality") == "FAI").select(
    ["participant", "condition", "mean", "ci_half"]))



FAI variance check (should be 0 in current run, non-zero after rerun):


participant,condition,mean,ci_half
str,str,f64,f64
"""EV_002""","""NEG""",-0.324764,0.0
"""EV_002""","""NEU""",-0.326973,0.0
"""EV_002""","""POS""",-0.238901,0.0
"""EV_003""","""NEG""",-0.226355,0.0
"""EV_003""","""NEU""",-0.216702,0.0
…,…,…,…
"""EV_007""","""NEU""",-0.086945,0.0
"""EV_007""","""POS""",-0.071133,0.0
"""EV_008""","""NEG""",NaN,NaN
